In [2]:
import pandas as pd

def wide_to_long(wide: pd.DataFrame, value_name: str) -> pd.DataFrame:
    """
    Convert wide df (index=date, columns=tickers) -> long df with columns:
    ['date','ticker', value_name]
    Works even if index/columns have odd names.
    """
    if wide is None or wide.empty:
        return pd.DataFrame(columns=["date", "ticker", value_name])

    s = wide.stack(dropna=False)  # MultiIndex Series (date, ticker)
    out = s.reset_index()
    # out has 3 columns: [date_level, ticker_level, 0]
    out.columns = ["date", "ticker", value_name]
    out["date"] = pd.to_datetime(out["date"])
    return out

def combine_sources_to_one_df_fixed(
    tickers: list[str],
    years: int = 20,
    end: str | None = None,
) -> pd.DataFrame:
    import yfinance as yf
    from pandas_datareader import data as pdr
    import numpy as np

    end_dt = pd.to_datetime(end) if end else pd.Timestamp.today().normalize()
    start = (end_dt - pd.DateOffset(years=years)).date().isoformat()

    # ---------- Yahoo ----------
    y = yf.download(
        tickers=tickers,
        start=start,
        end=end,
        auto_adjust=False,
        group_by="column",
        progress=False,
        threads=True,
    )
    if isinstance(y.columns, pd.MultiIndex):
        yahoo = y["Adj Close"].copy()
    else:
        # single ticker case
        yahoo = y[["Adj Close"]].rename(columns={"Adj Close": tickers[0]})

    yahoo.index = pd.to_datetime(yahoo.index)
    yahoo = yahoo.sort_index()

    # ---------- Stooq (best-effort; symbol mapping may be needed) ----------
    stooq_dict = {}
    for tkr in tickers:
        try:
            sym = tkr.lower()  # may not work for .OL; you'll need mapping for full coverage
            s_df = pdr.DataReader(sym, "stooq").sort_index()
            s = s_df["Close"].rename(tkr)
            s = s.loc[s.index >= pd.to_datetime(start)]
            stooq_dict[tkr] = s
        except Exception:
            continue

    stooq = pd.concat(stooq_dict.values(), axis=1) if stooq_dict else pd.DataFrame(index=yahoo.index)
    stooq.index = pd.to_datetime(stooq.index)
    stooq = stooq.sort_index()

    # ---------- Union dates + coalesce ----------
    all_dates = yahoo.index.union(stooq.index)
    yahoo = yahoo.reindex(all_dates)
    stooq = stooq.reindex(all_dates)

    consolidated = yahoo.combine_first(stooq)

    # source_used (priority: yahoo then stooq)
    source_used = pd.DataFrame(index=all_dates, columns=tickers, dtype="object")
    source_used[yahoo.notna()] = "yahoo"
    source_used[(source_used.isna()) & (stooq.notna())] = "stooq"

    # ---------- Build ONE long df ----------
    long = wide_to_long(consolidated, "price")
    long_src = wide_to_long(source_used, "source_used")  # works because it's “wide” too
    long = long.merge(long_src, on=["date", "ticker"], how="left")

    # Optional debug columns: per-source prices
    long = long.merge(wide_to_long(yahoo, "price_yahoo"), on=["date", "ticker"], how="left")
    long = long.merge(wide_to_long(stooq, "price_stooq"), on=["date", "ticker"], how="left")

    long = long.sort_values(["ticker", "date"]).reset_index(drop=True)
    return long

# ---- run ----
obx_tickers = ["EQNR.OL", "DNB.OL", "NHY.OL", "TEL.OL", "ORK.OL"]
df_all = combine_sources_to_one_df_fixed(obx_tickers, years=20)

print(df_all.head())
print(df_all["source_used"].value_counts(dropna=False))

        date  ticker      price source_used  price_yahoo price_stooq
0 2006-02-20  DNB.OL  31.417110       yahoo    31.417110         NaN
1 2006-02-21  DNB.OL  32.009880       yahoo    32.009880         NaN
2 2006-02-22  DNB.OL  31.911093       yahoo    31.911093         NaN
3 2006-02-23  DNB.OL  32.207481       yahoo    32.207481         NaN
4 2006-02-24  DNB.OL  32.108685       yahoo    32.108685         NaN
source_used
yahoo    25145
Name: count, dtype: int64


C:\Users\jonas\AppData\Local\Temp\ipykernel_29076\1091266816.py:12: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  s = wide.stack(dropna=False)  # MultiIndex Series (date, ticker)
C:\Users\jonas\AppData\Local\Temp\ipykernel_29076\1091266816.py:12: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  s = wide.stack(dropna=False)  # MultiIndex Series (date, ticker)
C:\Users\jonas\AppData\Local\Temp\ipykernel_29076\1091266816.py:12: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pand

In [4]:
import requests
import pandas as pd

def get_current_obx_tickers_yahoo() -> list[str]:
    url = "https://en.wikipedia.org/wiki/OBX_Index"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                      "(KHTML, like Gecko) Chrome/122.0 Safari/537.36"
    }
    html = requests.get(url, headers=headers, timeout=30).text
    tables = pd.read_html(html)

    # find table with a ticker column
    target = None
    for t in tables:
        cols = [str(c).lower() for c in t.columns]
        if any("ticker" in c for c in cols):
            target = t
            break
    if target is None:
        raise RuntimeError("Could not find OBX constituents table.")

    # pick the ticker column
    ticker_col = None
    for c in target.columns:
        if "ticker" in str(c).lower():
            ticker_col = c
            break

    raw = target[ticker_col].astype(str).tolist()

    # extract last token and append .OL
    tickers = []
    for x in raw:
        parts = x.replace("\xa0", " ").replace(":", " ").split()
        if parts:
            sym = parts[-1].strip()
            if sym and sym.lower() != "nan":
                tickers.append(sym + ".OL")

    # de-dup preserve order
    out, seen = [], set()
    for t in tickers:
        if t not in seen:
            out.append(t)
            seen.add(t)
    return out

print(get_current_obx_tickers_yahoo()[:10])

['AKRBP.OL', 'BWLPG.OL', 'DNB.OL', 'EQNR.OL', 'FRO.OL', 'GJF.OL', 'GOGL.OL', 'HAFNI.OL', 'HAUTO.OL', 'KOG.OL']


C:\Users\jonas\AppData\Local\Temp\ipykernel_29076\3082066675.py:11: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(html)


In [6]:
import pandas as pd

def missing_report_per_ticker(prices_wide: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for tkr in prices_wide.columns:
        s = prices_wide[tkr]

        # Per-ticker coverage window (where data exists)
        first_valid = s.first_valid_index()
        last_valid = s.last_valid_index()

        if first_valid is None:
            rows.append({
                "ticker": tkr,
                "rows_in_df": int(len(s)),
                "n_valid": 0,
                "n_missing": int(s.isna().sum()),
                "missing_pct_full_df": float(s.isna().mean()),
                "first_valid_date": None,
                "last_valid_date": None,
                "n_missing_within_valid_window": None,
                "missing_pct_within_valid_window": None,
                "longest_missing_streak_days": None,
            })
            continue

        window = s.loc[first_valid:last_valid]
        n_missing_window = int(window.isna().sum())
        n_window = int(len(window))
        missing_pct_window = float(n_missing_window / n_window) if n_window else None

        # Longest missing streak within valid window (in trading days, not calendar days)
        is_missing = window.isna().astype(int)
        # streak length = consecutive 1s
        streak = (is_missing.groupby((is_missing != is_missing.shift()).cumsum()).cumsum() * is_missing).max()
        longest_streak = int(streak) if pd.notna(streak) else 0

        rows.append({
            "ticker": tkr,
            "rows_in_df": int(len(s)),
            "n_valid": int(s.notna().sum()),
            "n_missing": int(s.isna().sum()),
            "missing_pct_full_df": float(s.isna().mean()),
            "first_valid_date": first_valid.date().isoformat(),
            "last_valid_date": last_valid.date().isoformat(),
            "n_missing_within_valid_window": n_missing_window,
            "missing_pct_within_valid_window": missing_pct_window,
            "longest_missing_streak_days": longest_streak,
        })

    rep = pd.DataFrame(rows)
    rep = rep.sort_values(
        ["missing_pct_within_valid_window", "missing_pct_full_df", "longest_missing_streak_days"],
        ascending=[False, False, False],
        na_position="last"
    ).reset_index(drop=True)
    return rep

In [ ]:
missing_report = missing_report_per_ticker(prices_wide)
print(missing_report.head(25))